# NEXUS — train the dMaSIF surface net on ALREADY-BUILT clouds from Drive

The surface build (marching-cubes + APBS) is the slow part — a few seconds of CPU **per complex**. Once a cloud is
built and saved, training on it is fast. This notebook is the **train-only** path: it loads the clouds you already
built into `MyDrive/nexus_cache/clouds` and trains + hold-out-validates the geodesic surface net on them. **No fetch,
no APBS, no rebuild** — so it needs almost no setup and starts training in seconds.

Point it at whatever you've processed so far (e.g. the 2,150 clouds) and read the held-out interface AUC. The trained
weights are saved back to `MyDrive/nexus_cache/nexus_dmasif.pt` (right next to the clouds), so they survive the session.

## 1 · GPU (Runtime → Change runtime type → GPU)

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, '|', torch.cuda.get_device_name(0) if device=='cuda' else 'CPU (training is small; fine either way)')

## 2 · Minimal deps (no APBS/pdb2pqr needed — we only train)
Training needs just torch + scikit-learn + scipy, which Colab already ships. This is a no-op if they're present.

In [ ]:
import sys, subprocess
subprocess.run(sys.executable+' -m pip -q install scikit-learn scipy 2>/dev/null', shell=True)
print('deps ok')

## 3 · Get the code

In [ ]:
import os, sys
BRANCH='claude/vectorize-gex-propensity-zp09w8'
if not os.path.isdir('/content/cell'):
    os.system(f'git clone -q -b {BRANCH} https://github.com/nikku03/cell.git /content/cell')
else:
    os.system('cd /content/cell && git pull -q')
sys.path.insert(0,'/content/cell/colab')
print('code:', 'ok' if os.path.isdir('/content/cell/colab') else 'MISSING')

## 4 · Mount Drive and count the clouds you've built

In [ ]:
from google.colab import drive
import glob
drive.mount('/content/drive')
CLOUDS = '/content/drive/MyDrive/nexus_cache/clouds'   # where the build/auto-save cell saved them
n = len(glob.glob(CLOUDS + '/*.pkl'))
print(f'{n} cached clouds in {CLOUDS}')
assert n >= 2, 'need at least 2 built clouds — check the path above'

## 5 · Train + hold out — score on exactly the clouds you have
Trains the dMaSIF geodesic net on a **75% split held out by complex** (no leakage — test complexes are never trained on)
and reports the held-out interface-discrimination AUC. `epochs=60` is plenty; bump it if the AUC is still climbing.
The weights are saved to `MyDrive/nexus_cache/nexus_dmasif.pt`.

In [ ]:
import nexus_train, importlib; importlib.reload(nexus_train)
res = nexus_train.train_from_cache(CLOUDS, epochs=60, device=device)
print('\ncomplexes trained on:', res.get('n_usable'),
      '| train/test:', f"{res.get('n_train')}/{res.get('n_test')}",
      '| held-out dMaSIF AUC:', res.get('dmasif_heldout_auc'))

## What this tells you
- **held-out dMaSIF AUC** — how well the surface net separates true interface point-pairs from non-interface pairs on
  complexes it never saw. On the full ~260-complex run it reached ~0.90 (MaSIF-grade); this run tells you where your
  current 2,150-cloud set lands.
- The build keeps running in your other notebook; whenever you have more clouds in Drive, just **re-run cell 5** — it
  reloads whatever is there and retrains. No rebuild.
- Trained weights: `MyDrive/nexus_cache/nexus_dmasif.pt` (also mirrored to `outputs/orphan/nexus_train_from_cache.json`
  in the repo checkout for the run summary).